In [50]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"
import os
import json
import random
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics.pairwise import cosine_similarity

Unsupervised

In [94]:
# constraints/load_allowed_labels.py
import json

def load_allowed_set(path="/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/allowed_labels_bosch.json") -> set[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    labels = data.get("labels", []) or []
    labels = {x.strip() for x in labels if isinstance(x, str) and x.strip()}
    return labels


In [95]:
# -------------------------
# CONFIG
# -------------------------
TRAIN_DATA_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/train_clean.json"
VALIDATION_DATA_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/val_clean.json"
TTP_DESC_PATH   = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/ttp_descriptions2.json"
BASE_MODEL = "ehsanaghaei/SecureBERT"

BATCH_SIZE = 32
PATIENCE   = 5
SEED       = 42

MAX_LEN_SENT = 192
MAX_LEN_TTP  = 256

LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_EPOCHS = 50
TEMPERATURE = 0.05

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# REPRODUCIBILITY
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# -------------------------
# LOAD DATA
# -------------------------
with open(TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(TTP_DESC_PATH, "r", encoding="utf-8") as f:
    ttp_descriptions = json.load(f)
allowed_set = load_allowed_set()  # set[str]

# -------------------------
# BUILD TRAIN PAIRS
# -------------------------
# only non empty ttps
train_pairs: List[Tuple[str, str]] = []
for item in train_data:
    sent = item["sentence"].strip()
    for ttp in item["labels"]:
        if ttp in ttp_descriptions and ttp in allowed_set:
            ttp_text = f"{ttp}: {ttp_descriptions[ttp]}"
            train_pairs.append((sent, ttp_text))

print(f"Training pairs: {len(train_pairs)}")
print("train pair sample:", train_pairs[0])
print("train pair sample:", train_pairs[1])

from collections import defaultdict, Counter
import random
import numpy as np

def normalize_label_list(lbls):
    out = []
    for x in (lbls or []):
        if isinstance(x, str):
            x = x.strip()
            if x:
                out.append(x)
    return out

# Build sentence -> set(labels) but only allowed labels
sent2labels = defaultdict(set)

for item in train_data:
    sent = (item.get("sentence") or "").strip()
    if not sent:
        continue
    lbls = normalize_label_list(item.get("labels"))
    for ttp in lbls:
        if ttp in allowed_set:
            sent2labels[sent].add(ttp)

# Keep only sentences that still have at least 1 allowed label
items = [(sent, sorted(lblset)) for sent, lblset in sent2labels.items() if lblset]

print("Unique sentences after filtering:", len(items))

# Compute "real" (unbalanced) label distribution on this filtered pool
real_counts = Counter()
for _, lbls in items:
    real_counts.update(lbls)

print("Allowed labels present in filtered pool:", len(real_counts))


Training pairs: 10764
train pair sample: ('FunnyDream can check system time to help determine when changes were made to specified files.', "T1124: An adversary may gather the system time and/or time zone settings from a local or remote system. The system time is set and stored by services, such as the Windows Time Service on Windows or <code>systemsetup</code> on macOS.(Citation: MSDN System Time)(Citation: Technet Windows Time Service)(Citation: systemsetup mac time) These time settings may also be synchronized between systems and services in an enterprise network, typically accomplished with a network time server within a domain.(Citation: Mac Time Sync)(Citation: linux system time)  System time information may be gathered in a number of ways, such as with [Net](https://attack.mitre.org/software/S0039) on Windows by performing <code>net time \\\\hostname</code> to gather the system time on a remote system. The victim's time zone may also be inferred from the current system time or ga

In [96]:
def make_multilabel_val_split(items, real_counts, val_frac=0.10, seed=42):
    rng = random.Random(seed)

    # Targets per label (what we want to see in validation)
    target = {lbl: int(round(val_frac * c)) for lbl, c in real_counts.items()}
    # Ensure at least 1 if label exists and val_frac > 0 (optional)
    for lbl, c in real_counts.items():
        if c > 0 and val_frac > 0:
            target[lbl] = max(target[lbl], 1)

    remaining = dict(target)

    # Shuffle items for randomness
    items_shuf = items[:]
    rng.shuffle(items_shuf)

    val_items = []
    train_items = []

    def score(lbls):
        # How much this sentence helps remaining quotas
        return sum(1 for l in lbls if remaining.get(l, 0) > 0)

    # Greedy: keep picking helpful sentences until quotas are mostly met
    for sent, lbls in items_shuf:
        s = score(lbls)
        # Take if it helps, or if we still have very few val samples and need to start
        if s > 0:
            val_items.append((sent, lbls))
            for l in lbls:
                if remaining.get(l, 0) > 0:
                    remaining[l] -= 1
        else:
            train_items.append((sent, lbls))

    # If val became too big (can happen), optionally trim by random / low contribution
    # Here we just return as-is.
    return train_items, val_items, target, remaining

train_items, val_items, target, remaining = make_multilabel_val_split(
    items, real_counts, val_frac=0.10, seed=SEED
)

print("Train sentences:", len(train_items))
print("Val sentences  :", len(val_items))

val_counts = Counter()
for _, lbls in val_items:
    val_counts.update(lbls)

print("Real counts (top5):", real_counts.most_common(5))
print("Val  counts (top5):", val_counts.most_common(5))


Train sentences: 9005
Val sentences  : 1012
Real counts (top5): [('T1027', 835), ('T1105', 817), ('T1140', 780), ('T1082', 497), ('T1083', 429)]
Val  counts (top5): [('T1027', 84), ('T1105', 82), ('T1140', 78), ('T1082', 51), ('T1083', 43)]


In [97]:
import math
from collections import Counter
from collections import Counter, defaultdict
import random, math
import numpy as np
def split_train_val_sentence_level(items, val_frac=0.10, seed=42):
    rng = random.Random(seed)
    items_shuf = items[:]
    rng.shuffle(items_shuf)
    n_val = int(round(val_frac * len(items_shuf)))
    val_items = items_shuf[:n_val]
    train_items = items_shuf[n_val:]
    return train_items, val_items

train_items_unbal, val_items = split_train_val_sentence_level(items, val_frac=0.10, seed=SEED)

print("Train sentences (unbal):", len(train_items_unbal))
print("Val sentences:", len(val_items), "-> wrote:", VALIDATION_DATA_PATH)

def get_sentence(it):
    # supports {"sentence":..., "labels":...} OR (sentence, labels) OR (sentence, labels, ...)
    if isinstance(it, dict):
        return it["sentence"]
    if isinstance(it, (list, tuple)):
        return it[0]
    raise TypeError(f"Unsupported item type: {type(it)}")

def get_labels(it):
    if isinstance(it, dict):
        return it["labels"]
    if isinstance(it, (list, tuple)):
        return it[1]
    raise TypeError(f"Unsupported item type: {type(it)}")

def set_labels(it, new_labels):
    # return same “shape” as input
    if isinstance(it, dict):
        out = dict(it)
        out["labels"] = new_labels
        return out
    if isinstance(it, (list, tuple)):
        # keep first element as sentence, second as labels, keep extras if any
        if len(it) == 2:
            return (it[0], new_labels)
        return (it[0], new_labels, *it[2:])
    raise TypeError(f"Unsupported item type: {type(it)}")

import math
import numpy as np
from collections import Counter

def compute_targets(
    counts: Counter,
    ref_quantile: float = 0.75,
    cap_mult: float = 1.5,
    floor_mult: float = 0.5,
    alpha: float = 0.6,
    min_count: int = 5,
):
    """
    Tempered rebalancing targets that allow BOTH oversampling and downsampling.

    target_l = clamp( ceil(c_l^alpha * ref^(1-alpha)), floor, cap )

    - alpha in (0,1): closer to 1 => gentler change; closer to 0 => stronger pull to ref
    - ref: quantile of counts (e.g., 0.75)
    - cap_mult: max target relative to ref (prevents exploding tail)
    - floor_mult: min target relative to ref (prevents destroying head via heavy downsampling)
    - min_count: absolute minimum target (keeps very rare labels from being downsampled to 0/1)
    """
    vals = np.array(list(counts.values()), dtype=float)
    ref = float(np.quantile(vals, ref_quantile))

    cap = float(ref * cap_mult)
    floor = float(ref * floor_mult)

    targets = {}
    for lbl, c in counts.items():
        c = float(c)

        # tempered pull toward ref
        t = (c ** alpha) * (ref ** (1.0 - alpha))
        t = math.ceil(t)

        # clamp to floor/cap + absolute min_count
        t = max(t, int(min_count), int(math.floor(floor)))
        t = min(t, int(math.ceil(cap)))

        targets[lbl] = int(t)

    return targets, ref, cap, floor


def tempered_oversample_sentences(train_items, targets, seed=42):
    rng = random.Random(seed)

    counts = Counter()
    for it in train_items:
        counts.update(get_labels(it))

    need = {lbl: max(0, targets[lbl] - counts.get(lbl, 0)) for lbl in targets}

    label2idx = defaultdict(list)
    for idx, it in enumerate(train_items):
        for lbl in get_labels(it):
            if lbl in need:
                label2idx[lbl].append(idx)

    augmented = list(train_items)

    max_iters = sum(need.values()) + 1000
    iters = 0
    while True:
        iters += 1
        if iters > max_iters:
            break

        lbl, remaining = max(need.items(), key=lambda x: x[1])
        if remaining <= 0:
            break

        candidates = label2idx.get(lbl, [])
        if not candidates:
            need[lbl] = 0
            continue

        pick_idx = rng.choice(candidates)
        picked = train_items[pick_idx]
        augmented.append(picked)

        for l in get_labels(picked):
            if l in need and need[l] > 0:
                need[l] -= 1

    return augmented


# counts on train (unbalanced) pool
train_counts = Counter()
for it in train_items_unbal:
    train_counts.update(get_labels(it))

targets, ref, cap, floor = compute_targets(
    train_counts,
    ref_quantile=0.75,
    cap_mult=1.5,
    floor_mult=0.6,
    alpha=0.6,
    min_count=6,
)
print(ref, cap, floor)

train_items_less_unbal = tempered_oversample_sentences(train_items_unbal, targets, seed=SEED)

# Inspect effect
new_counts = Counter()
for it in train_items_less_unbal:
    new_counts.update(get_labels(it))


# print("Reference (q=0.75):", ref_used, "Cap:", cap_used)
print("Original train label min/max:", min(train_counts.values()), max(train_counts.values()))
print("New train label min/max     :", min(new_counts.values()), max(new_counts.values()))


Train sentences (unbal): 9015
Val sentences: 1002 -> wrote: /home/simonettos/thijs/data_augmentatio_stefano/mitre/val_clean.json
70.0 105.0 42.0
Original train label min/max: 4 756
New train label min/max     : 42 758


In [98]:
def iter_sentence_label_items(sentence_items):
    """Yields (sentence, labels_list) from either dict-items or tuple-items."""
    for it in sentence_items:
        if isinstance(it, dict):
            sent = (it.get("sentence") or "").strip()
            lbls = it.get("labels", []) or []
        else:
            # tuple/list: (sent, labels, ...)
            sent = (it[0] or "").strip()
            lbls = it[1] if len(it) > 1 else []
        if not sent:
            continue
        # normalize labels
        out = []
        for x in (lbls or []):
            if isinstance(x, str):
                x = x.strip()
                if x:
                    out.append(x)
        if out:
            yield sent, out

print(len(allowed_set), "allowed labels for training pairs.")
source_train_items = train_items_less_unbal  

train_pairs: List[Tuple[str, str]] = []
for sent, lbls in iter_sentence_label_items(source_train_items):
    for ttp in lbls:
        if ttp in allowed_set and ttp in ttp_descriptions:
            ttp_text = f"{ttp}: {ttp_descriptions[ttp]}"
            train_pairs.append((sent, ttp_text))

print(f"Training pairs (from balanced sentences): {len(train_pairs)}")

print("Sample train pair 0:", train_pairs[0])

112 allowed labels for training pairs.
Training pairs (from balanced sentences): 11418
Sample train pair 0: ('Unusual inbound email activity where attachments or embedded URLs are delivered to users followed by execution of new processes or suspicious document behavior. Detection involves correlating email metadata, file creation, and network activity after a phishing message is received.', 'T1566: Adversaries may send phishing messages to gain access to victim systems. All forms of phishing are electronically delivered social engineering. Phishing can be targeted, known as spearphishing. In spearphishing, a specific individual, company, or industry will be targeted by the adversary. More generally, adversaries can conduct non-targeted phishing, such as in mass malware spam campaigns.  Adversaries may send victims emails containing malicious attachments or links, typically to execute malicious code on victim systems. Phishing may also be conducted via third-party services, like social 

In [99]:
# 1) Basic sanity: how many items, how many yielded by iterator?
print("source_train_items:", len(source_train_items))

n_yield = 0
n_labels_total = 0
first_few = []
for sent, lbls in iter_sentence_label_items(source_train_items):
    n_yield += 1
    n_labels_total += len(lbls)
    if len(first_few) < 3:
        first_few.append((sent[:80], lbls[:10]))

print("iter_sentence_label_items yielded:", n_yield)
print("total labels across yielded items:", n_labels_total)
print("first few yields:", first_few)

# 2) Check overlap between your labels and the filters
labels_in_data = set()
for _, lbls in iter_sentence_label_items(source_train_items):
    labels_in_data.update(lbls)

print("unique labels in data:", len(labels_in_data))
print("overlap with allowed_set:", len(labels_in_data & allowed_set))
print("overlap with ttp_descriptions:", len(labels_in_data & set(ttp_descriptions.keys())))
print("overlap with BOTH:", len(labels_in_data & allowed_set & set(ttp_descriptions.keys())))

# 3) Show a few labels that are being rejected (helpful to see formatting issues)
rejected_allowed = list(labels_in_data - allowed_set)[:20]
rejected_desc = list(labels_in_data - set(ttp_descriptions.keys()))[:20]
print("example labels not in allowed_set:", rejected_allowed)
print("example labels not in ttp_descriptions:", rejected_desc)



source_train_items: 10464
iter_sentence_label_items yielded: 10464
total labels across yielded items: 11418
first few yields: [('Unusual inbound email activity where attachments or embedded URLs are delivered ', ['T1566']), ('RTM can scan victim drives to look for specific banking software on the machine ', ['T1518']), ('Royal has been spread through the use of phishing campaigns including "call back', ['T1566'])]
unique labels in data: 111
overlap with allowed_set: 111
overlap with ttp_descriptions: 111
overlap with BOTH: 111
example labels not in allowed_set: []
example labels not in ttp_descriptions: []


In [100]:
with open(VALIDATION_DATA_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)   # list of {"sentence":..., "labels":[...]}

# -------------------------
# DATASET + COLLATE
# -------------------------
class PairDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]]):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx: int):
        return self.pairs[idx]  # (sent, ttp_text)

@dataclass
class DualCollator:
    tokenizer: object
    max_len_sent: int
    max_len_ttp: int

    def __call__(self, batch: List[Tuple[str, str]]) -> Dict[str, Dict[str, torch.Tensor]]:
        sents = [b[0] for b in batch]
        ttps  = [b[1] for b in batch]

        tok_sent = self.tokenizer(
            sents,
            padding=True,
            truncation=True,
            max_length=self.max_len_sent,
            return_tensors="pt",
        )
        tok_ttp = self.tokenizer(
            ttps,
            padding=True,
            truncation=True,
            max_length=self.max_len_ttp,
            return_tensors="pt",
        )
        return {"sent": tok_sent, "ttp": tok_ttp}

# -------------------------
# MODEL: SecureBERT bi-encoder
# -------------------------
class SecureBertEmbedder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)

    @staticmethod
    def mean_pool(last_hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
        summed = (last_hidden * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-9)
        return summed / denom

    def encode_batch(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        emb = self.mean_pool(out.last_hidden_state, attention_mask)
        emb = F.normalize(emb, p=2, dim=1)
        return emb

    @torch.no_grad()
    def encode_texts(self, tokenizer, texts: List[str], batch_size: int, max_length: int) -> np.ndarray:
        self.eval()
        all_embs = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i+batch_size]
            tok = tokenizer(
                chunk,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            tok = {k: v.to(DEVICE) for k, v in tok.items()}
            embs = self.encode_batch(tok["input_ids"], tok["attention_mask"])
            all_embs.append(embs.detach().cpu().numpy())
        return np.vstack(all_embs)

# -------------------------
# LOSS: In-batch negatives
# -------------------------
class InBatchNegativesLoss(nn.Module):
    def __init__(self, temperature: float = 0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, emb_a: torch.Tensor, emb_b: torch.Tensor) -> torch.Tensor:
        logits = (emb_a @ emb_b.t()) / self.temperature
        labels = torch.arange(logits.size(0), device=logits.device)
        return F.cross_entropy(logits, labels)

# -------------------------
# TOKENIZER / DATALOADER
# -------------------------
# IMPORTANT: use AutoTokenizer for SecureBERT
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

train_ds = PairDataset(train_pairs)
collator = DualCollator(tokenizer=tokenizer, max_len_sent=MAX_LEN_SENT, max_len_ttp=MAX_LEN_TTP)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=collator
)

# -------------------------
# INIT MODEL / OPT / SCHED
# -------------------------
model = SecureBertEmbedder(BASE_MODEL).to(DEVICE)
loss_fn = InBatchNegativesLoss(temperature=TEMPERATURE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# -------------------------
# PREP VAL TTP TEXTS
# -------------------------
ttp_ids = [t for t in allowed_set if t in ttp_descriptions]
ttp_texts = [f"{t}: {ttp_descriptions[t]}" for t in ttp_ids]
# -------------------------
# VALIDATION FUNCTION
# -------------------------
def get_val_sentence(item):
    if isinstance(item, dict):
        return (item.get("sentence") or "").strip()
    if isinstance(item, (list, tuple)):
        return (item[0] or "").strip()
    raise TypeError(f"Unsupported val item type: {type(item)}")

def get_val_labels(item):
    if isinstance(item, dict):
        return item.get("labels", []) or []
    if isinstance(item, (list, tuple)):
        return item[1] if len(item) > 1 else []
    raise TypeError(f"Unsupported val item type: {type(item)}")

def validate(model: SecureBertEmbedder, k_list=(1, 5, 10)) -> Dict[str, float]:
    model.eval()

    ttp_embs = model.encode_texts(tokenizer, ttp_texts, batch_size=64, max_length=MAX_LEN_TTP)

    hits_at_k = {k: 0 for k in k_list}
    mean_recall_at_k = {k: 0.0 for k in k_list}
    mrr = 0.0
    used = 0

    for item in val_data:
        sent = get_val_sentence(item)
        raw_lbls = get_val_labels(item)

        gold = {t.strip() for t in raw_lbls if isinstance(t, str) and t.strip()}
        gold = {t for t in gold if t in allowed_set and t in ttp_descriptions}
        if not sent or not gold:
            continue

        used += 1

        sent_emb = model.encode_texts(tokenizer, [sent], batch_size=1, max_length=MAX_LEN_SENT)[0]
        sims = cosine_similarity(sent_emb.reshape(1, -1), ttp_embs)[0]
        ranking = np.argsort(-sims)

        rr = 0.0
        for rank_pos, idx in enumerate(ranking, start=1):
            if ttp_ids[idx] in gold:
                rr = 1.0 / rank_pos
                break
        mrr += rr

        for k in k_list:
            topk = [ttp_ids[i] for i in ranking[:k]]
            hits_at_k[k] += int(any(t in gold for t in topk))
            mean_recall_at_k[k] += len(set(topk) & gold) / max(1, len(gold))

    n = max(1, used)
    metrics = {f"hit@{k}": hits_at_k[k] / n for k in k_list}
    metrics.update({f"mean_recall@{k}": mean_recall_at_k[k] / n for k in k_list})
    metrics["mrr"] = mrr / n
    metrics["val_used"] = used
    return metrics



# -------------------------
# SAVE HELPERS
# -------------------------
def save_model(model: SecureBertEmbedder, tokenizer, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    model.backbone.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = SecureBertEmbedder(BASE_MODEL).to(DEVICE)

metrics = validate(model, k_list=(1, 5, 10))
print("UNSUPERVISED baseline (pretrained SecureBERT bi-encoder)")
print(", ".join([f"{k}: {v:.4f}" for k, v in metrics.items()]))

Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


UNSUPERVISED baseline (pretrained SecureBERT bi-encoder)
hit@1: 0.0379, hit@5: 0.0979, hit@10: 0.1499, mean_recall@1: 0.0366, mean_recall@5: 0.0949, mean_recall@10: 0.1455, mrr: 0.0918, val_used: 1134.0000


In [86]:
from collections import Counter

in_sent_level = set()
in_pairs = set()
missing_desc = set()
not_allowed = set()

# Optional: count how often each label appears
sent_label_counts = Counter()

for sent, lbls in iter_sentence_label_items(source_train_items):
    for ttp in lbls:
        if ttp in allowed_set:
            in_sent_level.add(ttp)
            sent_label_counts[ttp] += 1
            if ttp in ttp_descriptions:
                in_pairs.add(ttp)
            else:
                missing_desc.add(ttp)
        else:
            not_allowed.add(ttp)

print(f"Sentence-level unique labels (allowed_set): {len(in_sent_level)}")
print(f"Pair-usable unique labels (also in ttp_descriptions): {len(in_pairs)}")
print(f"Allowed but MISSING in ttp_descriptions: {len(missing_desc)}")
print(f"Not in allowed_set (should be 0 if source is already filtered): {len(not_allowed)}")

# Show the top missing ones by frequency
top_missing = sorted(missing_desc, key=lambda t: sent_label_counts[t], reverse=True)[:30]
print("\nTop missing_desc labels (by frequency in train sentences):")
for t in top_missing:
    print(t, "count_in_sentences:", sent_label_counts[t])

# Quick check: are you missing sub-techniques because of formatting?
print("\nExamples of missing_desc labels:", sorted(list(missing_desc))[:25])
print("Example keys in ttp_descriptions:", list(ttp_descriptions.keys())[:10])


Sentence-level unique labels (allowed_set): 111
Pair-usable unique labels (also in ttp_descriptions): 111
Allowed but MISSING in ttp_descriptions: 0
Not in allowed_set (should be 0 if source is already filtered): 0

Top missing_desc labels (by frequency in train sentences):

Examples of missing_desc labels: []
Example keys in ttp_descriptions: ['T1055.011', 'T1053.005', 'T1205.002', 'T1066', 'T1560.001', 'T1021.005', 'T1047', 'T1156', 'T1113', 'T1027.011']
